# 00 · Setup: the whole Tinker API on one slide

Three primitives. Every method in this webinar is a different arrangement of them.

| call | what it does |
|---|---|
| `training_client.forward_backward(data, loss_fn)` | compute loss + gradients (on Tinker's GPUs) |
| `training_client.optim_step(adam_params)` | apply the update |
| `sampling_client.sample(prompt, ...)` | generate from current weights |

You write the loop on your laptop; Tinker runs the GPUs.

In [ ]:
from common import *

# What can we train?
for m in tinker.RestClient().get_server_capabilities().result().supported_models:
    print(m.model_name)

In [ ]:
training_client = new_training_client()
sampling_client = training_client.save_weights_and_get_sampling_client().result()
print("Training + sampling clients ready for", MODEL)

## Baseline: what does the model do before we train it?

In [ ]:
show("raw text completion", generate(sampling_client, "To be, or not to be,", chat=False, temperature=0.7))
show("chat prompt", generate(sampling_client, "What's a good name for a pet goldfish?"))
show("math prompt", generate(sampling_client, gsm8k_convo(load_gsm8k("test")[0]["question"]), max_tokens=MAX_TOKENS))

In [ ]:
# Our task for the day: grade-school math (GSM8K) answered correctly *within 256 tokens*.
# The model knows the math but rambles, so many answers run out of tokens before \boxed{}.
# We'll come back to this score after every section.
eval_gsm8k(sampling_client, label="0. base model")